#  Training ViT Transfomer on Fire Detection for outdoor Scenario

## Overview
In this notebook, we train a **SWIN-T** model on a custom dataset designed for detecting **fire scenarios** in **outdoor images**. This dataset represents one of the specific scenarios for our **Mixture of Experts (MoE)** model, where each expert specializes in a different scenario (e.g., fire detection in outdoor, indoor, satellite, or far-field environments).


##  Environment & Setup

In [ ]:
import cv2
import os

# This script resizes all images in the train and valid directories to 224x224 pixels.

for img_file in os.listdir("train/images"):
    path = os.path.join("train/images", img_file)
    img = cv2.imread(path)
    resized = cv2.resize(img, (224, 224))
    cv2.imwrite(path, resized)
    
for img_file in os.listdir("valid/images"):
    path = os.path.join("valid/images", img_file)
    img = cv2.imread(path)
    resized = cv2.resize(img, (224, 224))
    cv2.imwrite(path, resized)


In [1]:
# 1. Install required libraries
!pip install timm torchvision albumentations pycocotools --quiet


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


##  Data Loading & Preprocessing


In [ ]:
import os
import random
import shutil

# Paths
test_images_dir = 'test/images'
test_labels_dir = 'test/labels'
val_images_dir = 'valid/images'
val_labels_dir = 'valid/labels'

os.makedirs(val_images_dir, exist_ok=True)
os.makedirs(val_labels_dir, exist_ok=True)

random.seed(42)
test_images = [f for f in os.listdir(test_images_dir) if f.endswith(('.jpg', '.png'))]
val_images = random.sample(test_images, 300)

for img_file in val_images:
    label_file = os.path.splitext(img_file)[0] + '.txt'
    shutil.move(os.path.join(test_images_dir, img_file), os.path.join(val_images_dir, img_file))
    label_path = os.path.join(test_labels_dir, label_file)
    if os.path.exists(label_path):
        shutil.move(label_path, os.path.join(val_labels_dir, label_file))

print(f"Moved {len(val_images)} images and labels to validation.")


Moved 300 images and their labels to the validation set.


## Checking GPU Availability

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Current device:", torch.cuda.get_device_name(0))

## Dataset & Preprocessing

## Load Dataloaders

## Load Swin Transformer with Detection Head

## Train the Swin Expert

In [ ]:
from torch.utils.data import Dataset, DataLoader
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import os
import torch
import torch.nn as nn
from torchvision.models.detection import FasterRCNN
from torchvision.ops import MultiScaleRoIAlign
from torchvision.models.detection.rpn import AnchorGenerator
from timm import create_model
from torch.optim import AdamW
from tqdm import tqdm

class FireDetectionDataset(Dataset):
    def __init__(self, images_dir, labels_dir, transform=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.image_files = [f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png'))]
        self.transform = transform

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.image_files[idx])
        label_path = os.path.join(self.labels_dir, os.path.splitext(self.image_files[idx])[0] + '.txt')

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w, _ = image.shape

        boxes = []
        labels = []
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls, cx, cy, bw, bh = map(float, parts)
                    # Convert to absolute coordinates first
                    x1 = (cx - bw / 2) * w
                    y1 = (cy - bh / 2) * h
                    x2 = (cx + bw / 2) * w
                    y2 = (cy + bh / 2) * h
                    boxes.append([x1, y1, x2, y2])
                    labels.append(1)

        # Handle missing boxes
        if not boxes:
            boxes = [[0, 0, 1, 1]]  # Small dummy box
            labels = [0]

        # Apply augmentation (this will scale boxes to 224x224)
        if self.transform:
            augmented = self.transform(image=image, bboxes=boxes, labels=labels)
            image = augmented['image']
            boxes = augmented['bboxes']
            labels = augmented['labels']

        # Ensure we have valid boxes after augmentation
        if not boxes:
            boxes = [[0, 0, 1, 1]]
            labels = [0]

        # Convert to tensors
        target = {
            'boxes': torch.tensor(boxes, dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64)
        }

        return image, target

    def __len__(self):
        return len(self.image_files)

class SwinBackbone(nn.Module):
    def __init__(self, out_channels=256):
        super().__init__()
        self.body = create_model('swin_tiny_patch4_window7_224', pretrained=True, features_only=True)
        self.out_channels = out_channels

        # Channel sizes from Swin-T outputs
        self.fpn_input_channels = [96, 192, 384, 768]

        # 1x1 conv to unify output channels for FPN compatibility
        self.lateral_convs = nn.ModuleList([
            nn.Conv2d(in_c, out_channels, kernel_size=1)
            for in_c in self.fpn_input_channels
        ])

    def forward(self, x):
        feats = self.body(x)  # List of 4 outputs
        fpn_feats = {}
        for idx, feat in enumerate(feats):
            # Swin outputs are (B, H, W, C), convert to (B, C, H, W)
            if feat.dim() == 4 and feat.shape[-1] == self.fpn_input_channels[idx]:
                feat = feat.permute(0, 3, 1, 2)
            fpn_feats[str(idx)] = self.lateral_convs[idx](feat)
        return fpn_feats

# Custom Identity Transform to bypass Faster R-CNN's resizing
class IdentityTransform(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, images, targets=None):
        from torchvision.models.detection.image_list import ImageList
        
        # Images are already preprocessed
        if isinstance(images, list):
            # Convert list of tensors to batch tensor
            batch_images = torch.stack(images)
        else:
            batch_images = images
        
        # Create image_sizes list (all images are 224x224)
        batch_size = batch_images.shape[0]
        image_sizes = [(224, 224) for _ in range(batch_size)]
        
        # Create proper ImageList object
        image_list = ImageList(batch_images, image_sizes)
        
        if targets is not None:
            return image_list, targets
        else:
            return image_list
    
    def postprocess(self, result, image_shapes, original_image_sizes):
        """
        Postprocess the output of the model.
        Since we're not resizing images, we can return results as-is.
        """
        # In our case, image_shapes and original_image_sizes are the same (224x224)
        # so we can return the results without modification
        return result

def get_swin_fasterrcnn(num_classes=2):
    backbone = SwinBackbone(out_channels=256)

    anchor_generator = AnchorGenerator(
        sizes=((16, 32), (32, 64), (64, 128), (128, 256)),  # One tuple per feature map
        aspect_ratios=((0.5, 1.0, 2.0), (0.5, 1.0, 2.0), (0.5, 1.0, 2.0), (0.5, 1.0, 2.0))  # One tuple per feature map
    )

    roi_pooler = MultiScaleRoIAlign(
        featmap_names=["0", "1", "2", "3"],
        output_size=7,
        sampling_ratio=2
    )

    model = FasterRCNN(
        backbone=backbone,
        num_classes=num_classes,
        rpn_anchor_generator=anchor_generator,
        box_roi_pool=roi_pooler,
        transform=None  # *** Disable transform completely ***
    )
    
    # Replace the transform with our identity transform
    model.transform = IdentityTransform()

    return model

# Data setup - Remove normalization since we're bypassing the transform
transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

train_dataset = FireDetectionDataset("train/images", "train/labels", transform)
val_dataset = FireDetectionDataset("valid/images", "valid/labels", transform)

def collate_fn(batch):
    """Custom collate function to handle batch processing"""
    images, targets = tuple(zip(*batch))
    
    # Stack images into a batch tensor
    images = torch.stack(images, dim=0)
    
    # Keep targets as list (required by Faster R-CNN)
    return images, list(targets)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)

# Model setup
model = get_swin_fasterrcnn(num_classes=2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# Training loop
num_epochs = 10
model.train()

print("Starting training...")
for epoch in range(num_epochs):
    total_loss = 0
    successful_batches = 0
    
    for batch_idx, (images, targets) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")):
        try:
            # Move to device
            images = images.to(device)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            # Debug: Check shapes
            if batch_idx == 0:
                print(f"Batch images shape: {images.shape}")
                print(f"Number of targets: {len(targets)}")
                for i, target in enumerate(targets):
                    print(f"Target {i} boxes shape: {target['boxes'].shape}")
            
            # Forward pass
            loss_dict = model(images, targets)
            loss = sum(loss for loss in loss_dict.values())
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            successful_batches += 1
            
        except Exception as e:
            print(f"Error in batch {batch_idx}: {e}")
            import traceback
            traceback.print_exc()
            continue

    if successful_batches > 0:
        avg_loss = total_loss / successful_batches
        print(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_loss:.4f}, Successful batches: {successful_batches}/{len(train_loader)}")
    else:
        print(f"Epoch {epoch+1}/{num_epochs}, No successful batches!")

# Save the model
torch.save(model.state_dict(), "swin_frcnn_outdoor.pt")
print("Model saved as swin_frcnn_outdoor_fixed.pt")

Starting training...


Epoch 1/10:   0%|          | 0/45 [00:00<?, ?it/s]

Batch images shape: torch.Size([4, 3, 224, 224])
Number of targets: 4
Target 0 boxes shape: torch.Size([1, 4])
Target 1 boxes shape: torch.Size([1, 4])
Target 2 boxes shape: torch.Size([1, 4])
Target 3 boxes shape: torch.Size([2, 4])


Epoch 1/10: 100%|██████████| 45/45 [04:20<00:00,  5.79s/it]


Epoch 1/10, Average Loss: 0.5770, Successful batches: 45/45


...

Epoch 9/10, Average Loss: 0.3170, Successful batches: 45/45


Epoch 10/10:   0%|          | 0/45 [00:00<?, ?it/s]

Batch images shape: torch.Size([4, 3, 224, 224])
Number of targets: 4
Target 0 boxes shape: torch.Size([1, 4])
Target 1 boxes shape: torch.Size([1, 4])
Target 2 boxes shape: torch.Size([1, 4])
Target 3 boxes shape: torch.Size([1, 4])


Epoch 10/10: 100%|██████████| 45/45 [03:26<00:00,  4.59s/it]


Epoch 10/10, Average Loss: 0.3040, Successful batches: 45/45
Model saved as swin_frcnn_outdoor_fixed.pt


## Run Inference on Test Images

In [ ]:
model = get_swin_fasterrcnn(num_classes=2)
model.load_state_dict(torch.load("swin_frcnn_outdoor.pt", map_location=device))
model.to(device)
model.eval()
print("Model loaded and ready for inference.")

test_dataset = FireDetectionDataset("test/images", "test/labels", transform)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

import matplotlib.pyplot as plt

for i, (image, _) in enumerate(test_loader):
    image = image[0].to(device)
    with torch.no_grad():
        prediction = model([image])[0]

    boxes = prediction['boxes'].cpu().numpy()
    scores = prediction['scores'].cpu().numpy()

    # Show results
    img_np = image.permute(1, 2, 0).cpu().numpy()
    plt.figure(figsize=(10, 10))
    plt.imshow(img_np)
    for box, score in zip(boxes, scores):
        if score > 0.5:
            x1, y1, x2, y2 = box
            plt.gca().add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1,
                                              edgecolor='red', facecolor='none', linewidth=2))
            plt.text(x1, y1, f"{score:.2f}", color='white', fontsize=12,
                     bbox=dict(facecolor='red', edgecolor='none', pad=1))
    plt.axis('off')
    plt.show()

    if i == 4: break  # Show only first 5
